# Multi-Pipeline Workflow: From Market Data to Risk Report

## Introduction

This notebook demonstrates how to **chain multiple processing steps** together to create an end-to-end workflow:

1. **Build market data** (realistic curves, vol surface)
2. **Construct portfolio** from configuration
3. **Price the portfolio**
4. **Run scenario analysis**
5. **Generate risk report**

---

### Workflow Flow

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Build Market   │ ──▶ │ Build Portfolio │ ──▶ │  Price Portfolio│
│     Data        │     │                 │     │                 │
└─────────────────┘     └─────────────────┘     └─────────────────┘
         │                       │                       │
         ▼                       ▼                       ▼
   [Market State]        [Portfolio State]       [Pricing Results]
                                                         │
                                                         ▼
                              ┌─────────────────────────────────────┐
                              │         Run Scenarios               │
                              │    (Stress Test Portfolio)          │
                              └─────────────────────────────────────┘
                                                         │
                                                         ▼
                                              [Scenario Report]
```

In [ ]:
# =============================================================================
# SETUP: Imports
# =============================================================================

import sys
from pathlib import Path
from datetime import date
import numpy as np

sys.path.insert(0, str(Path.cwd().parents[1]))

# Market data components
from src.marketdata.core.market import Market
from src.marketdata.core.ids import MarketId
from src.marketdata.core.interfaces import Quote
from src.marketdata.curves.term_structure import ZeroRateCurve
from src.marketdata.surfaces.vol_surface import GridVolSurface

# Portfolio components
from src.portfolio.core import Portfolio, Position
from src.instruments.fx.options.vanilla import FxVanillaEuropeanOption

# Pricing
from src.pricers.fx.european_bsm import FxVanillaEuropeanOptionBsmPricer

# Plotting
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')

print("All imports successful!")

## Step 1: Build Market Data

First, we create the market environment with **realistic** data:
- Spot quotes
- Zero rate curves (term structure)
- Volatility surfaces (smile and term structure)

In [ ]:
# =============================================================================
# STEP 1: Build Market Data with Realistic Term Structures
# =============================================================================

print("="*70)
print("Step 1: Building Market Data")
print("="*70)

# Define market IDs
spot_id = MarketId.parse("FX.SPOT.EURUSD")
vol_id = MarketId.parse("FX.VOL.EURUSD")
usd_curve_id = MarketId.parse("IR.ZERO.USD")
eur_curve_id = MarketId.parse("IR.ZERO.EUR")

# Base market parameters
BASE_SPOT = 1.0850

# Realistic USD zero rate curve (inverted - typical 2024+ environment)
tenors = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
usd_rates = np.array([0.0540, 0.0535, 0.0520, 0.0480, 0.0460, 0.0440, 0.0435, 0.0430, 0.0425, 0.0420])
eur_rates = np.array([0.0380, 0.0385, 0.0390, 0.0395, 0.0400, 0.0405, 0.0408, 0.0410, 0.0412, 0.0415])

# Realistic vol surface (expiry x strike grid)
vol_expiries = np.array([0.083, 0.167, 0.25, 0.5, 1.0])  # 1M, 2M, 3M, 6M, 1Y
vol_strikes = np.array([0.95, 0.98, 1.00, 1.02, 1.05, 1.08, 1.10, 1.12, 1.15]) * BASE_SPOT
vol_grid = np.array([
    [0.115, 0.100, 0.088, 0.085, 0.090, 0.095, 0.100, 0.108, 0.118],
    [0.112, 0.098, 0.087, 0.084, 0.088, 0.093, 0.098, 0.105, 0.115],
    [0.110, 0.096, 0.086, 0.084, 0.087, 0.092, 0.096, 0.103, 0.112],
    [0.108, 0.095, 0.086, 0.085, 0.088, 0.092, 0.095, 0.100, 0.108],
    [0.105, 0.094, 0.088, 0.087, 0.090, 0.093, 0.096, 0.100, 0.106],
])

# Build market
market = Market(
    asof=date.today(),
    quotes={spot_id: Quote(value=BASE_SPOT)},
    curves={
        usd_curve_id: ZeroRateCurve(tenors=tenors, zero_rates=usd_rates),
        eur_curve_id: ZeroRateCurve(tenors=tenors, zero_rates=eur_rates),
    },
    vols={
        vol_id: GridVolSurface(
            expiries=vol_expiries,
            strikes=vol_strikes,
            implied_vols=vol_grid,
        ),
    },
)

print(f"\n  Market built successfully")
print(f"  As of:       {market.asof}")
print(f"  Spot:        EURUSD = {BASE_SPOT}")
print(f"  USD Curve:   Inverted (3M={usd_rates[0]:.2%}, 30Y={usd_rates[-1]:.2%})")
print(f"  EUR Curve:   Normal (3M={eur_rates[0]:.2%}, 30Y={eur_rates[-1]:.2%})")
print(f"  Vol Surface: {vol_grid.shape[0]} expiries x {vol_grid.shape[1]} strikes")

## Step 2: Build Portfolio

Construct a portfolio from a configuration specification.

In [ ]:
# =============================================================================
# STEP 2: Build Portfolio from Configuration
# =============================================================================

print("\n" + "="*70)
print("Step 2: Building Portfolio")
print("="*70)

# Portfolio configuration (as you might see in a YAML file)
portfolio_config = {
    'name': 'FX_Options_Demo_Book',
    'base_currency': 'USD',
    'positions': [
        {
            'id': 'LONG_CALL',
            'option_type': 'call',
            'strike': 1.10,
            'expiry': 0.25,
            'notional': 10_000_000,
            'direction': 'long',
        },
        {
            'id': 'SHORT_PUT',
            'option_type': 'put',
            'strike': 1.05,
            'expiry': 0.25,
            'notional': 10_000_000,
            'direction': 'short',
        },
        {
            'id': 'STRADDLE_CALL',
            'option_type': 'call',
            'strike': 1.085,
            'expiry': 0.5,
            'notional': 5_000_000,
            'direction': 'long',
        },
        {
            'id': 'STRADDLE_PUT',
            'option_type': 'put',
            'strike': 1.085,
            'expiry': 0.5,
            'notional': 5_000_000,
            'direction': 'long',
        },
    ]
}

# Build portfolio from config
def build_portfolio_from_config(config):
    """Build a Portfolio from configuration dictionary."""
    positions = []
    
    for pos_cfg in config['positions']:
        quantity = 1 if pos_cfg['direction'] == 'long' else -1
        
        instrument = FxVanillaEuropeanOption(
            option_type=pos_cfg['option_type'],
            notional=pos_cfg['notional'],
            strike=pos_cfg['strike'],
            expiry=pos_cfg['expiry'],
            spot_id=spot_id,
            vol_id=vol_id,
            domestic_curve_id=usd_curve_id,
            foreign_curve_id=eur_curve_id,
        )
        
        positions.append(Position(
            position_id=pos_cfg['id'],
            instrument=instrument,
            quantity=quantity,
        ))
    
    return Portfolio(positions=positions)

portfolio = build_portfolio_from_config(portfolio_config)

print(f"\n  Portfolio built: {portfolio_config['name']}")
print(f"  Positions: {len(portfolio)}")
print("\n  Position Details:")
for pos in portfolio:
    inst = pos.instrument
    dir_str = "LONG" if pos.quantity > 0 else "SHORT"
    print(f"    - {pos.position_id}: {dir_str} {inst.option_type.upper()} "
          f"K={inst.strike:.4f} T={inst.expiry}Y N={inst.notional:,.0f}")

## Step 3: Price Portfolio

Compute present value and Greeks for all positions.

In [ ]:
# =============================================================================
# STEP 3: Price the Portfolio
# =============================================================================

print("\n" + "="*70)
print("Step 3: Pricing Portfolio")
print("="*70)

pricer = FxVanillaEuropeanOptionBsmPricer()

# Price each position
pricing_results = {}
total_pv = 0
total_greeks = {'delta': 0, 'gamma': 0, 'vega': 0, 'theta': 0}

for pos in portfolio:
    # Get PV and Greeks separately (correct API usage)
    pv = pricer.price(pos.instrument, market) * pos.quantity
    greeks = pricer.greeks(pos.instrument, market)
    
    pricing_results[pos.position_id] = {
        'pv': pv,
        'delta': greeks.get('delta', 0) * pos.quantity,
        'gamma': greeks.get('gamma', 0) * pos.quantity,
        'vega': greeks.get('vega', 0) * pos.quantity,
        'theta': greeks.get('theta', 0) * pos.quantity,
    }
    
    total_pv += pv
    for greek in total_greeks:
        total_greeks[greek] += pricing_results[pos.position_id][greek]

# Display results
print(f"\n  Pricing completed")
print(f"\n  {'Position':<18} {'PV':>15} {'Delta':>12} {'Gamma':>12} {'Vega':>12} {'Theta':>12}")
print(f"  {'-'*82}")

for pos_id, results in pricing_results.items():
    print(f"  {pos_id:<18} ${results['pv']:>13,.0f} {results['delta']:>12,.0f} "
          f"{results['gamma']:>12,.0f} {results['vega']:>12,.0f} {results['theta']:>12,.0f}")

print(f"  {'-'*82}")
print(f"  {'TOTAL':<18} ${total_pv:>13,.0f} {total_greeks['delta']:>12,.0f} "
      f"{total_greeks['gamma']:>12,.0f} {total_greeks['vega']:>12,.0f} {total_greeks['theta']:>12,.0f}")

## Step 4: Run Scenario Analysis

Define and execute stress scenarios.

In [ ]:
# =============================================================================
# STEP 4: Define and Run Scenarios
# =============================================================================

print("\n" + "="*70)
print("Step 4: Running Scenario Analysis")
print("="*70)

def create_shocked_market(spot_shock=0, vol_shock=0, rate_shock=0):
    """Create a market with applied shocks."""
    shocked_spot = BASE_SPOT * (1 + spot_shock)
    shocked_vol_grid = np.maximum(0.01, vol_grid + vol_shock)
    shocked_usd_rates = usd_rates + rate_shock
    shocked_eur_rates = eur_rates + rate_shock
    
    return Market(
        asof=date.today(),
        quotes={spot_id: Quote(value=shocked_spot)},
        curves={
            usd_curve_id: ZeroRateCurve(tenors=tenors, zero_rates=shocked_usd_rates),
            eur_curve_id: ZeroRateCurve(tenors=tenors, zero_rates=shocked_eur_rates),
        },
        vols={
            vol_id: GridVolSurface(
                expiries=vol_expiries,
                strikes=vol_strikes * (1 + spot_shock),  # Adjust strikes for spot move
                implied_vols=shocked_vol_grid,
            ),
        },
    )

# Scenario definitions
scenarios = [
    {'name': 'Base Case', 'spot_shock': 0, 'vol_shock': 0},
    {'name': 'Spot +5%', 'spot_shock': 0.05, 'vol_shock': 0},
    {'name': 'Spot -5%', 'spot_shock': -0.05, 'vol_shock': 0},
    {'name': 'Vol +5pts', 'spot_shock': 0, 'vol_shock': 0.05},
    {'name': 'Vol -5pts', 'spot_shock': 0, 'vol_shock': -0.05},
    {'name': 'Risk-Off (Spot -5%, Vol +5pts)', 'spot_shock': -0.05, 'vol_shock': 0.05},
    {'name': 'Risk-On (Spot +5%, Vol -2pts)', 'spot_shock': 0.05, 'vol_shock': -0.02},
    {'name': 'Crisis (Spot -15%, Vol +15pts)', 'spot_shock': -0.15, 'vol_shock': 0.15},
]

# Run scenarios
scenario_results = []

for scenario in scenarios:
    # Create shocked market
    shocked_market = create_shocked_market(
        spot_shock=scenario['spot_shock'],
        vol_shock=scenario['vol_shock'],
    )
    
    # Price portfolio
    scenario_pv = 0
    for pos in portfolio:
        pv = pricer.price(pos.instrument, shocked_market) * pos.quantity
        scenario_pv += pv
    
    pnl = scenario_pv - total_pv
    
    scenario_results.append({
        'name': scenario['name'],
        'spot': BASE_SPOT * (1 + scenario['spot_shock']),
        'pv': scenario_pv,
        'pnl': pnl,
        'pnl_pct': pnl / abs(total_pv) * 100 if total_pv != 0 else 0,
    })

# Display results
print(f"\n  {'Scenario':<40} {'Spot':>10} {'PV':>15} {'P&L':>15}")
print(f"  {'-'*82}")

for result in scenario_results:
    print(f"  {result['name']:<40} {result['spot']:>10.4f} "
          f"${result['pv']:>13,.0f} ${result['pnl']:>+13,.0f}")

print(f"  {'-'*82}")

## Step 5: Generate Risk Report

Create a comprehensive summary with visualizations.

In [ ]:
# =============================================================================
# STEP 5: Generate Risk Report
# =============================================================================

print("\n" + "="*70)
print("Step 5: Risk Report")
print("="*70)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Scenario P&L
ax1 = axes[0, 0]
scenario_names = [r['name'] for r in scenario_results[1:]]  # Exclude base case
pnls = [r['pnl'] / 1000 for r in scenario_results[1:]]
colors = ['green' if p > 0 else 'red' for p in pnls]

bars = ax1.barh(scenario_names, pnls, color=colors, alpha=0.7, edgecolor='black')
ax1.axvline(x=0, color='black', linewidth=1)
ax1.set_xlabel('P&L (USD thousands)')
ax1.set_title('Scenario P&L Impact', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Plot 2: Greek Exposure
ax2 = axes[0, 1]
greeks = list(total_greeks.keys())
greek_values = [total_greeks[g] / 1000 for g in greeks]
bar_colors = ['blue', 'green', 'purple', 'orange']

ax2.bar(greeks, greek_values, color=bar_colors, alpha=0.7, edgecolor='black')
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_ylabel('Value (USD thousands)')
ax2.set_title('Portfolio Greek Exposure', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Plot 3: Position-level PV breakdown
ax3 = axes[1, 0]
pos_names = list(pricing_results.keys())
pos_pvs = [pricing_results[p]['pv'] / 1000 for p in pos_names]
pos_colors = ['green' if pv > 0 else 'red' for pv in pos_pvs]

ax3.bar(pos_names, pos_pvs, color=pos_colors, alpha=0.7, edgecolor='black')
ax3.axhline(y=0, color='black', linewidth=1)
ax3.set_ylabel('PV (USD thousands)')
ax3.set_title('Position-Level PV', fontweight='bold')
ax3.tick_params(axis='x', rotation=45)
ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Spot sensitivity
ax4 = axes[1, 1]
spot_range = np.linspace(BASE_SPOT * 0.90, BASE_SPOT * 1.10, 30)
spot_pnls = []

for spot in spot_range:
    temp_market = create_shocked_market(spot_shock=(spot/BASE_SPOT - 1))
    pv = sum(pricer.price(pos.instrument, temp_market) * pos.quantity 
             for pos in portfolio)
    spot_pnls.append((pv - total_pv) / 1000)

ax4.plot(spot_range, spot_pnls, 'b-', linewidth=2)
ax4.fill_between(spot_range, spot_pnls, 0, where=(np.array(spot_pnls) > 0), 
                  alpha=0.3, color='green')
ax4.fill_between(spot_range, spot_pnls, 0, where=(np.array(spot_pnls) < 0), 
                  alpha=0.3, color='red')
ax4.axhline(y=0, color='black', linewidth=1)
ax4.axvline(x=BASE_SPOT, color='gray', linestyle='--', alpha=0.7)
ax4.set_xlabel('EURUSD Spot')
ax4.set_ylabel('P&L (USD thousands)')
ax4.set_title('Spot Sensitivity', fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.suptitle(f"Risk Report: {portfolio_config['name']}\nAs of {date.today()}", 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Summary Report
# =============================================================================

print("\n" + "="*70)
print("EXECUTIVE SUMMARY")
print("="*70)

print(f"\n Portfolio: {portfolio_config['name']}")
print(f" As of:     {date.today()}")

print(f"\n VALUATION")
print(f" {'-'*40}")
print(f" Portfolio PV:     ${total_pv:>15,.0f}")
print(f" Number of trades:         {len(portfolio):>5}")

print(f"\n RISK SUMMARY")
print(f" {'-'*40}")
print(f" Total Delta:      ${total_greeks['delta']:>15,.0f}")
print(f" Total Gamma:      {total_greeks['gamma']:>16,.0f}")
print(f" Total Vega:       ${total_greeks['vega']:>15,.0f}")
print(f" Total Theta:      ${total_greeks['theta']:>15,.0f} /day")

# Worst/best scenarios
pnl_sorted = sorted(scenario_results[1:], key=lambda x: x['pnl'])
worst = pnl_sorted[0]
best = pnl_sorted[-1]

print(f"\n SCENARIO ANALYSIS")
print(f" {'-'*40}")
print(f" Worst case:  {worst['name']:<30}")
print(f"              P&L: ${worst['pnl']:>+15,.0f}")
print(f" Best case:   {best['name']:<30}")
print(f"              P&L: ${best['pnl']:>+15,.0f}")

print(f"\n RISK INTERPRETATION")
print(f" {'-'*40}")
if total_greeks['delta'] > 0:
    print(f" -> Portfolio is LONG EURUSD (benefits from EUR strength)")
else:
    print(f" -> Portfolio is SHORT EURUSD (benefits from EUR weakness)")

if total_greeks['vega'] > 0:
    print(f" -> Portfolio is LONG volatility (benefits from vol increase)")
else:
    print(f" -> Portfolio is SHORT volatility (benefits from vol decrease)")

print(f" -> Daily time decay: ${abs(total_greeks['theta']):,.0f}")

print("\n" + "="*70)

## Key Takeaways

### Realistic Market Data
- **Term structures** affect forward prices and discounting differently at each expiry
- **Vol surfaces** capture smile/skew effects that flat assumptions miss
- Scenarios should shift full surfaces, not just single values

### Workflow Benefits
- **Modularity**: Each step is independently testable
- **Reproducibility**: Same config → same results
- **Auditability**: Full logging and artifact trail
- **Flexibility**: Easily swap components

### Next Steps
- Add more asset classes (rates, equities)
- Implement delta hedging step
- Add P&L attribution pipeline
- Create batch processing for multiple portfolios